<a href="https://colab.research.google.com/github/DAN-jpg-byte/2025_10_18_git_study_on_youtube/blob/main/image_to_pdf_converter_mono.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pillow-heif

import os
import shutil
from datetime import datetime
from PIL import Image, ImageOps
import pillow_heif

# Googleドライブのマウント
from google.colab import drive
drive.mount('/content/drive')

# HEIC対応
pillow_heif.register_heif_opener()

# --- フォルダパスの設定 ---
# 大元のフォルダ
BASE_DIR = '/content/drive/MyDrive/Aマウント用【iphoneの画像をPDF本化】'

# 各種フォルダ
INPUT_DIR = f'{BASE_DIR}/input_images'
OUTPUT_DIR = f'{BASE_DIR}/output_pdf'
PROCESSED_DIR = f'{BASE_DIR}/processed_images'

# 軽量化設定
TARGET_SIZE = 2500
PDF_QUALITY = 70

# --- フォルダの準備 ---
# 必要なフォルダが存在しない場合は自動作成
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# 処理対象の画像を取得（HEIC, JPG, PNG）
valid_exts = ('.heic', '.jpg', '.jpeg', '.png')
files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(valid_exts)]

# ファイル名順にソート（PDFのページ順を正しくするため）
image_paths = sorted([os.path.join(INPUT_DIR, f) for f in files])

if not image_paths:
    print(f"画像が見つかりません。以下のフォルダに処理したい写真を入れてから再度実行してください。\n -> {INPUT_DIR}")
else:
    print(f"{len(image_paths)}枚の画像が見つかりました。モノクロ化処理を開始します...")

    pdf_images = []

    for i, img_path in enumerate(image_paths):
        try:
            # 画像を開く
            img = Image.open(img_path)

            # Exif情報に基づいて画像の向きを正しく修正
            img = ImageOps.exif_transpose(img)

            # 一旦RGBモードに変換
            if img.mode != 'RGB':
                img = img.convert('RGB')

            # リサイズ処理
            width, height = img.size
            if max(width, height) > TARGET_SIZE:
                img.thumbnail((TARGET_SIZE, TARGET_SIZE), Image.Resampling.LANCZOS)

            # モノクロ（グレースケール）に変換
            img = img.convert('L')

            pdf_images.append(img)

            # 途中経過の表示
            if (i + 1) % 10 == 0:
                print(f"  -> {i + 1}枚目を処理完了...")

        except Exception as e:
            print(f"エラー: {os.path.basename(img_path)} の処理中に問題が発生しました - {e}")

    # PDFとして保存とファイルの移動処理
    if pdf_images:
        # タイムスタンプの取得（PDFファイル名と、処理済みフォルダ名に使用）
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        # PDFの保存
        output_pdf_path = os.path.join(OUTPUT_DIR, f"output_mono_{timestamp}.pdf")
        print(f"\nモノクロPDFを保存しています... (圧縮設定: Q={PDF_QUALITY})")

        pdf_images[0].save(
            output_pdf_path,
            save_all=True,
            append_images=pdf_images[1:],
            quality=PDF_QUALITY,
            optimize=True
        )
        print(f"✅ PDF保存完了: {output_pdf_path}")

        # --- 処理が終わった画像を processed_images へ移動 ---
        print("\n元の画像を処理済みフォルダへ移動します...")

        # 今回の処理用のサブフォルダを作成（例: batch_20260516_120526）
        batch_processed_dir = os.path.join(PROCESSED_DIR, f"batch_{timestamp}")
        os.makedirs(batch_processed_dir, exist_ok=True)

        for img_path in image_paths:
            # 移動先のファイルパスを作成
            dest_path = os.path.join(batch_processed_dir, os.path.basename(img_path))
            # ファイルを移動
            shutil.move(img_path, dest_path)

        print(f"✅ 画像の移動完了: {batch_processed_dir} に保管されました")
        print("インプットフォルダは空になり、次の処理の準備が整いました。")